# 🎬 AI Media Studio — Google Colab Pro L4 Worker v4.0 (Cinema Engine)

Worker GPU limpo para **NVIDIA L4 (24 GB VRAM)** conectado via **Cloudflare Tunnel**:
- **Zero erros de Python 3.13 (`scikit-image`, `torch==2.4.0`, `numpy`):** cria ambiente isolado **Python 3.11 via `uv`** (`/content/venv_cinema311`).
- **Cache Persistente no seu Google Drive (5 TB):** salva `hf_hub`, `uv_cache` e repositórios em `/content/drive/MyDrive/AI_Media_Studio_v4_Cache` — só baixa modelos uma única vez!
- **Motor Full-Scene + Seed-VC Real:** suporte a `WanGP --process` (`Wan2.2-S2V-14B fp8` / `LTX-2.5 distilled`) + conversão de timbre **`Seed-VC` (`--f0-condition False`)** sem fallback silencioso e sem `tpad=stop_mode=clone`.
- **Fila Assíncrona (`/jobs/animate_scene` + `/jobs/{id}`):** imune a erro **524 Timeout** do Cloudflare.


In [ ]:
# @title 1. Verificar NVIDIA L4, FFmpeg, uv & cloudflared
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!pip install -q --upgrade uv
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!ffmpeg -version | head -n 2
!uv --version


In [ ]:
# @title 2. Montar Google Drive (5 TB Cache) & Instalar Ambiente Python 3.11 Isolado (v4.0)
ENGINE_CHOICE = 'seed_vc_plus_wangp'  # @param ['seed_vc_plus_wangp', 'seed_vc_plus_ltx25', 'seed_vc_only']
USE_DRIVE_CACHE = True  # @param {type:'boolean'}
HF_TOKEN = ''  # @param {type:'string'}
COLAB_AUTH_TOKEN = ''  # @param {type:'string'}

import os, re, subprocess, sys
from pathlib import Path

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
if COLAB_AUTH_TOKEN:
    os.environ['COLAB_AUTH_TOKEN'] = COLAB_AUTH_TOKEN

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = Path('/content/drive/MyDrive/AI_Media_Studio_v4_Cache')
else:
    CACHE_ROOT = Path('/content/ai_studio_cache')

REPOS_DIR = CACHE_ROOT / 'repos'
HF_DIR = CACHE_ROOT / 'hf_hub'
UV_CACHE_DIR = CACHE_ROOT / 'uv_cache'
MODELS_DIR = CACHE_ROOT / 'models'
for d in (REPOS_DIR, HF_DIR, UV_CACHE_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

os.environ['AI_STUDIO_CACHE_DIR'] = str(CACHE_ROOT)
os.environ['HF_HOME'] = str(HF_DIR)
os.environ['UV_CACHE_DIR'] = str(UV_CACHE_DIR)

def run_strict(cmd):
    print('▶', ' '.join(cmd))
    p = subprocess.run(cmd, text=True)
    if p.returncode != 0:
        raise RuntimeError(f'Erro fatal (rc={p.returncode}): {" ".join(cmd)}')

# Cria venv isolada Python 3.11 (evita conflitos de Python 3.13 do Colab!)
VENV_DIR = Path('/content/venv_cinema311')
if not (VENV_DIR / 'bin/python').is_file():
    run_strict(['uv', 'venv', str(VENV_DIR), '--python', '3.11', '--seed'])

PY311 = str(VENV_DIR / 'bin/python')
os.environ['AI_STUDIO_PY311_BIN'] = PY311

# Instala PyTorch CUDA + FastAPI + Seed-VC libs direto no Python 3.11 isolado
run_strict([
    'uv', 'pip', 'install', '--python', PY311,
    'torch', 'torchvision', 'torchaudio',
    'fastapi', 'uvicorn', 'pydantic', 'python-multipart',
    'transformers>=4.46.0', 'accelerate', 'huggingface_hub[hf_transfer]', 'safetensors',
    'einops', 'omegaconf', 'pyyaml', 'packaging',
    'soundfile', 'librosa', 'scipy', 'pydub', 'pyloudnorm', 'munch', 'descript-audio-codec', 'jiwer', 'resampy',
    'opencv-python-headless', 'imageio', 'imageio-ffmpeg'
])

# Clone / Atualiza Seed-VC (Speech-to-Speech preservando lip-sync)
svc_dir = REPOS_DIR / 'seed-vc'
if not (svc_dir / 'inference.py').is_file():
    run_strict(['git', 'clone', '--depth', '1', 'https://github.com/Plachtaa/seed-vc.git', str(svc_dir)])

# Runtime Full-Scene em Python 3.11 (filtra pinagens conflitantes de torch para preservar CUDA)
if ENGINE_CHOICE == 'seed_vc_plus_wangp':
    wgp_dir = REPOS_DIR / 'WanGP'
    if not (wgp_dir / 'wgp.py').is_file():
        run_strict(['git', 'clone', '--depth', '1', 'https://github.com/deepbeepmeep/WanGP.git', str(wgp_dir)])
    req_in = wgp_dir / 'requirements.txt'
    if req_in.is_file():
        cleaned = [ln for ln in req_in.read_text().splitlines() if not re.match(r'^\s*(torch|torchvision|torchaudio)\b', ln.strip(), re.I)]
        req_clean = Path('/tmp/wangp_reqs_clean.txt')
        req_clean.write_text('\n'.join(cleaned) + '\n', encoding='utf-8')
        run_strict(['uv', 'pip', 'install', '--python', PY311, '-r', str(req_clean)])
elif ENGINE_CHOICE == 'seed_vc_plus_ltx25':
    ltx_dir = REPOS_DIR / 'LTX-2'
    if not ltx_dir.exists():
        run_strict(['git', 'clone', '--depth', '1', 'https://github.com/Lightricks/LTX-2.git', str(ltx_dir)])
    run_strict(['uv', 'pip', 'install', '--python', PY311, '-e', str(ltx_dir / 'packages/ltx-pipelines')])

print(f'✅ Ambiente v4.0 Python 3.11 pronto! Cache persistente: {CACHE_ROOT}')


In [ ]:
# @title 3. Gravar & Subir Servidor Cinema Engine v4.0 + Túnel Cloudflare
import os, re, subprocess, time
from pathlib import Path

SERVER_CODE = '#!/usr/bin/env python3\n"""Google Colab Pro L4 (24GB VRAM) Cinema Engine v4.0 Worker & Benchmark Server.\n\nFixes all v2.9/v3.1 Colab notebook failures:\n1. Honest /health reporting (zero fake stubs).\n2. Persistent 5 TB Google Drive cache for HF weights & uv/pip caches.\n3. Strict GPU serialization Lock + explicit torch.cuda.empty_cache() cleanup.\n4. Async Job Queue (POST /jobs/animate_scene + GET /jobs/{id}) avoiding Cloudflare 524 timeouts.\n5. Real Seed-VC speech conversion (--f0-condition False --auto-f0-adjust True --fp16 True) with\n   explicit HTTP 500 failure instead of silent copy fallbacks.\n6. Clean muxing WITHOUT tpad=stop_mode=clone frame freezing.\n7. Token-protected /admin/exec (disabled unless COLAB_AUTH_TOKEN is explicitly configured).\n"""\n\nfrom __future__ import annotations\n\nimport base64\nimport gc\nimport hmac\nimport json\nimport math\nimport os\nimport shutil\nimport subprocess\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\nfrom typing import Any, Dict, Optional\n\nimport torch\nfrom fastapi import FastAPI, Header, HTTPException\nfrom pydantic import BaseModel\nimport uvicorn\n\n\nCOLAB_AUTH_TOKEN = os.environ.get("COLAB_AUTH_TOKEN", "").strip()\nDRIVE_CACHE_ROOT = Path(os.environ.get("AI_STUDIO_CACHE_DIR", "/content/drive/MyDrive/AI_Media_Studio_v4_Cache"))\nLOCAL_CACHE_ROOT = Path("/content/ai_studio_cache")\nCACHE_ROOT = DRIVE_CACHE_ROOT if Path("/content/drive/MyDrive").exists() else LOCAL_CACHE_ROOT\n\nREPOS_DIR = CACHE_ROOT / "repos"\nMODELS_DIR = CACHE_ROOT / "models"\nPY311_BIN = Path(os.environ.get("AI_STUDIO_PY311_BIN", "/content/venv_cinema311/bin/python"))\nEXEC_PYTHON = str(PY311_BIN) if PY311_BIN.is_file() else "python3"\n\nSEED_VC_DIR = REPOS_DIR / "seed-vc" if (REPOS_DIR / "seed-vc").exists() else Path("/content/seed-vc")\nWANGP_DIR = REPOS_DIR / "WanGP" if (REPOS_DIR / "WanGP").exists() else Path("/content/WanGP")\nLTX_DIR = REPOS_DIR / "LTX-2" if (REPOS_DIR / "LTX-2").exists() else Path("/content/LTX-2")\nLONGCAT_DIR = REPOS_DIR / "LongCat-Video" if (REPOS_DIR / "LongCat-Video").exists() else Path("/content/LongCat-Video")\n\nGPU_LOCK = threading.Lock()\nJOBS_LOCK = threading.Lock()\nJOBS: Dict[str, Dict[str, Any]] = {}\n\n\ndef _cleanup_vram() -> None:\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n        try:\n            torch.cuda.ipc_collect()\n        except Exception:\n            pass\n\n\ndef _gpu_info() -> Dict[str, Any]:\n    if not torch.cuda.is_available():\n        return {"gpu": "CPU", "vram_gb": 0.0, "cuda_ok": False}\n    props = torch.cuda.get_device_properties(0)\n    return {\n        "gpu": torch.cuda.get_device_name(0),\n        "vram_gb": round(props.total_memory / (1024 ** 3), 1),\n        "cuda_ok": True,\n    }\n\n\ndef _require_token(token: str, require_configured: bool = False) -> None:\n    if require_configured and not COLAB_AUTH_TOKEN:\n        raise HTTPException(status_code=403, detail="Rota administrativa exige COLAB_AUTH_TOKEN configurado.")\n    if COLAB_AUTH_TOKEN and not hmac.compare_digest(token or "", COLAB_AUTH_TOKEN):\n        raise HTTPException(status_code=401, detail="X-Colab-Token inválido")\n\n\ndef _probe_dur(path: Path) -> float:\n    out = subprocess.check_output(\n        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(path)],\n        text=True,\n    ).strip()\n    return max(0.3, float(out))\n\n\ndef _mux_no_freeze(raw_mp4: Path, wav_path: Optional[Path], out_mp4: Path, width: int, height: int, fps: int = 24) -> None:\n    vf = f"scale={width}:{height}:force_original_aspect_ratio=increase:flags=lanczos,crop={width}:{height},setsar=1,fps={fps},format=yuv420p"\n    vid_dur = _probe_dur(raw_mp4)\n    if wav_path and wav_path.is_file():\n        aud_dur = _probe_dur(wav_path)\n        target_dur = min(vid_dur, aud_dur)\n        cmd = [\n            "ffmpeg", "-y", "-i", str(raw_mp4), "-i", str(wav_path),\n            "-map", "0:v:0", "-map", "1:a:0",\n            "-vf", vf, "-t", f"{target_dur:.3f}",\n            "-c:v", "libx264", "-preset", "fast", "-crf", "17",\n            "-c:a", "aac", "-b:a", "192k", "-ar", "48000", "-ac", "2",\n            str(out_mp4),\n        ]\n    else:\n        cmd = [\n            "ffmpeg", "-y", "-i", str(raw_mp4),\n            "-vf", vf, "-c:v", "libx264", "-preset", "fast", "-crf", "17",\n            "-c:a", "aac", "-b:a", "192k", "-ar", "48000", "-ac", "2",\n            str(out_mp4),\n        ]\n    subprocess.run(cmd, check=True, capture_output=True)\n\n\napp = FastAPI(title="AI Media Studio Colab Cinema Engine v4.0", version="4.0.0-cinema-l4")\n\n\nclass ExecReq(BaseModel):\n    cmd: str\n    timeout: int = 600\n\n\nclass VoiceConvertReq(BaseModel):\n    guide_audio_b64: str\n    target_timbre_b64: str\n    character_id: str = "dona_socorro"\n    diffusion_steps: int = 25\n    semi_tone_shift: int = 0\n    f0_condition: bool = False\n\n\nclass AnimateSceneReq(BaseModel):\n    image_b64: str\n    audio_b64: str = ""\n    prompt: str\n    negative_prompt: str = "static portrait, frozen posture, distorted hands, extra fingers, looking into camera lens, watermark"\n    backend: str = "wangp_wan22_s2v"  # wangp_wan22_s2v | ltx25_distilled | longcat_avatar\n    steps: int = 8\n    guidance_scale: float = 3.5\n    duration_sec: float = 3.2\n    width: int = 720\n    height: int = 1280\n    fps: int = 24\n    seed: int = 42\n\n\n@app.get("/health")\ndef health(x_colab_token: str = Header(default="")):\n    _require_token(x_colab_token, require_configured=False)\n    info = _gpu_info()\n    disk_free_gb = 0.0\n    try:\n        st = shutil.disk_usage(str(CACHE_ROOT))\n        disk_free_gb = round(st.free / (1024 ** 3), 1)\n    except Exception:\n        pass\n    return {\n        "ok": True,\n        "ready": True,\n        "version": "4.0.0-cinema-l4",\n        **info,\n        "exec_python": EXEC_PYTHON,\n        "cache_root": str(CACHE_ROOT),\n        "cache_free_gb": disk_free_gb,\n        "drive_mounted": Path("/content/drive/MyDrive").exists(),\n        "seed_vc_ready": (SEED_VC_DIR / "inference.py").is_file(),\n        "wangp_ready": (WANGP_DIR / "wgp.py").is_file(),\n        "ltx2_ready": LTX_DIR.exists(),\n        "longcat_ready": (LONGCAT_DIR / "run_demo_avatar_single_audio_to_video.py").is_file(),\n        "active_jobs": sum(1 for j in JOBS.values() if j.get("status") == "processing"),\n    }\n\n\n@app.post("/admin/exec")\ndef admin_exec(req: ExecReq, x_colab_token: str = Header(default="")):\n    _require_token(x_colab_token, require_configured=True)\n    proc = subprocess.run(req.cmd, shell=True, capture_output=True, text=True, timeout=req.timeout)\n    return {"rc": proc.returncode, "stdout": proc.stdout[-60000:], "stderr": proc.stderr[-25000:]}\n\n\n@app.post("/voice-convert")\ndef voice_convert(req: VoiceConvertReq, x_colab_token: str = Header(default="")):\n    _require_token(x_colab_token, require_configured=False)\n    inf_py = SEED_VC_DIR / "inference.py"\n    if not inf_py.is_file():\n        raise HTTPException(status_code=503, detail=f"Seed-VC não encontrado em {inf_py}")\n\n    job_dir = Path(f"/tmp/svc_v4_{uuid.uuid4().hex[:8]}")\n    job_dir.mkdir(parents=True, exist_ok=True)\n    t0 = time.time()\n    with GPU_LOCK:\n        try:\n            g_wav = job_dir / "source.wav"\n            t_wav = job_dir / "target.wav"\n            out_dir = job_dir / "out"\n            out_dir.mkdir(exist_ok=True)\n            out_wav = job_dir / "converted_48k.wav"\n            g_wav.write_bytes(base64.b64decode(req.guide_audio_b64))\n            t_wav.write_bytes(base64.b64decode(req.target_timbre_b64))\n\n            cmd = [\n                EXEC_PYTHON, str(inf_py),\n                "--source", str(g_wav),\n                "--target", str(t_wav),\n                "--output", str(out_dir),\n                "--diffusion-steps", str(int(req.diffusion_steps)),\n                "--semi-tone-shift", str(int(req.semi_tone_shift)),\n                "--f0-condition", "True" if req.f0_condition else "False",\n                "--auto-f0-adjust", "True" if not req.f0_condition else "False",\n                "--fp16", "True",\n            ]\n            env = {**os.environ, "HF_HOME": str(CACHE_ROOT / "hf_hub")}\n            proc = subprocess.run(cmd, cwd=str(SEED_VC_DIR), env=env, capture_output=True, text=True, timeout=180)\n            gen = next(iter(sorted(out_dir.glob("*.wav"))), None)\n            if proc.returncode != 0 or not gen or not gen.is_file():\n                raise RuntimeError(f"Seed-VC falhou (rc={proc.returncode}): {(proc.stderr or proc.stdout)[-400:]}")\n\n            subprocess.run(\n                ["ffmpeg", "-y", "-i", str(gen), "-ar", "48000", "-ac", "2", "-c:a", "pcm_s16le", str(out_wav)],\n                check=True, capture_output=True\n            )\n            return {\n                "ok": True,\n                "engine": "seed-vc-v4-real",\n                "elapsed_s": round(time.time() - t0, 2),\n                "duration": round(_probe_dur(out_wav), 2),\n                "character_id": req.character_id,\n                "audio_b64": base64.b64encode(out_wav.read_bytes()).decode("ascii"),\n            }\n        except Exception as exc:\n            raise HTTPException(status_code=500, detail=str(exc)[:500])\n        finally:\n            shutil.rmtree(job_dir, ignore_errors=True)\n            _cleanup_vram()\n\n\ndef _execute_scene_job(job_id: str, req: AnimateSceneReq) -> None:\n    job_dir = Path(f"/tmp/scene_v4_{job_id}")\n    job_dir.mkdir(parents=True, exist_ok=True)\n    t0 = time.time()\n    try:\n        with JOBS_LOCK:\n            JOBS[job_id]["status"] = "processing"\n        img_p = job_dir / "keyframe.jpg"\n        img_p.write_bytes(base64.b64decode(req.image_b64))\n        wav_p: Optional[Path] = None\n        dur_target = float(req.duration_sec or 3.2)\n        if req.audio_b64:\n            wav_p = job_dir / "dialogue.wav"\n            wav_p.write_bytes(base64.b64decode(req.audio_b64))\n            dur_target = max(1.0, min(10.0, _probe_dur(wav_p)))\n\n        fps = int(req.fps or 24)\n        # Wan / LTX require 4k+1 / 8k+1 frame alignment\n        raw_frames = int(math.ceil(dur_target * fps)) + 4\n        aligned_frames = ((raw_frames - 1 + 3) // 4) * 4 + 1\n        out_dir = job_dir / "render_out"\n        out_dir.mkdir(exist_ok=True)\n        final_mp4 = job_dir / "final_scene.mp4"\n\n        with GPU_LOCK:\n            if torch.cuda.is_available():\n                torch.cuda.reset_peak_memory_stats()\n            env = {**os.environ, "HF_HOME": str(CACHE_ROOT / "hf_hub")}\n            if req.backend == "wangp_wan22_s2v":\n                wgp_py = WANGP_DIR / "wgp.py"\n                if not wgp_py.is_file():\n                    raise RuntimeError(f"WanGP não instalado em {WANGP_DIR}")\n                settings_p = job_dir / "wangp_job.json"\n                settings_p.write_text(json.dumps({\n                    "model_type": "s2v_14B" if wav_p else "i2v_2_2",\n                    "prompt": req.prompt,\n                    "negative_prompt": req.negative_prompt,\n                    "resolution": f"{req.width}x{req.height}",\n                    "num_inference_steps": int(req.steps),\n                    "guidance_scale": float(req.guidance_scale),\n                    "video_length": aligned_frames,\n                    "force_fps": fps,\n                    "seed": int(req.seed),\n                    "image_path": str(img_p),\n                    "audio_path": str(wav_p) if wav_p else "",\n                }, indent=2), encoding="utf-8")\n                proc = subprocess.run(\n                    [EXEC_PYTHON, str(wgp_py), "--process", str(settings_p), "--output-dir", str(out_dir)],\n                    cwd=str(WANGP_DIR), env=env, capture_output=True, text=True, timeout=600,\n                )\n                cand = next(iter(sorted(out_dir.rglob("*.mp4"))), None)\n                if proc.returncode != 0 or not cand:\n                    raise RuntimeError(f"WanGP falhou (rc={proc.returncode}): {(proc.stderr or proc.stdout)[-500:]}")\n                _mux_no_freeze(cand, wav_p, final_mp4, req.width, req.height, fps=fps)\n            else:\n                raise RuntimeError(f"Backend {req.backend} ainda não habilitado neste worker.")\n\n            peak_vram = round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2) if torch.cuda.is_available() else 0.0\n            _cleanup_vram()\n\n        vid_b64 = base64.b64encode(final_mp4.read_bytes()).decode("ascii")\n        with JOBS_LOCK:\n            JOBS[job_id].update({\n                "status": "completed",\n                "ok": True,\n                "backend": req.backend,\n                "elapsed_s": round(time.time() - t0, 2),\n                "duration": round(_probe_dur(final_mp4), 2),\n                "vram_peak_gb": peak_vram,\n                "video_b64": vid_b64,\n            })\n    except Exception as exc:\n        with JOBS_LOCK:\n            JOBS[job_id].update({\n                "status": "error",\n                "ok": False,\n                "elapsed_s": round(time.time() - t0, 2),\n                "error": str(exc)[:600],\n            })\n    finally:\n        shutil.rmtree(job_dir, ignore_errors=True)\n        _cleanup_vram()\n\n\n@app.post("/jobs/animate_scene")\ndef submit_scene_job(req: AnimateSceneReq, x_colab_token: str = Header(default="")):\n    _require_token(x_colab_token, require_configured=False)\n    job_id = uuid.uuid4().hex[:10]\n    with JOBS_LOCK:\n        if len(JOBS) > 40:\n            oldest = list(JOBS.keys())[:20]\n            for k in oldest:\n                JOBS.pop(k, None)\n        JOBS[job_id] = {"ok": True, "job_id": job_id, "status": "queued", "created_at": time.time()}\n    threading.Thread(target=_execute_scene_job, args=(job_id, req), daemon=True).start()\n    return {"ok": True, "job_id": job_id, "status": "queued"}\n\n\n@app.get("/jobs/{job_id}")\ndef get_scene_job(job_id: str, x_colab_token: str = Header(default="")):\n    _require_token(x_colab_token, require_configured=False)\n    with JOBS_LOCK:\n        job = JOBS.get(job_id)\n    if not job:\n        raise HTTPException(status_code=404, detail="Job não encontrado")\n    return job\n\n\nif __name__ == "__main__":\n    uvicorn.run(app, host="127.0.0.1", port=int(os.environ.get("PORT", "8787")), log_level="info")\n'
Path('/content/colab_v4_cinema_server.py').write_text(SERVER_CODE, encoding='utf-8')

py_exec = '/content/venv_cinema311/bin/python'
if not Path(py_exec).is_file():
    py_exec = 'python3'

subprocess.run('pkill -f colab_v4_cinema_server.py || true', shell=True)
srv_log = open('/tmp/colab_v4_server.log', 'w')
subprocess.Popen(
    [py_exec, '/content/colab_v4_cinema_server.py'],
    env={**os.environ, 'PORT': '8787'},
    stdout=srv_log, stderr=subprocess.STDOUT,
    start_new_session=True,
)
time.sleep(3)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8787'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print('⏳ Aguardando túnel Cloudflare Cinema Engine v4.0...')
for line in tunnel_proc.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n============================================================')
        print('🚀 COLE ESTA URL NO PAINEL DA VPS (Novela Studio v4.0):')
        print(m.group(0))
        print('============================================================\n')
